Project Overview

Study: Metagenomics Analysis_16S rRNA gene sequencing of matched fecal, mucosal, and tumor microbiota from colorectal cancer patients

# Amplicon Sequencing Analysis of Colorectal Cancer Microbiome Samples

## Project Description

This notebook presents a reproducible QIIME 2 workflow for the analysis of 16S rRNA amplicon sequencing data from human colorectal cancer samples. The objective is to characterize microbial community composition and diversity across different sample types collected from colorectal cancer patients.

The analysis includes data import, quality assessment, denoising with DADA2, feature table generation, taxonomic classification, diversity analysis, and visualization.

## Computational Environment

- Operating System: Ubuntu (WSL2)
- Environment Manager: Miniconda
- Microbiome Analysis Platform: QIIME 2 (2026.4)
- Interactive Analysis: Jupyter Notebook
  
## Dataset Details

- BioProject: PRJNA1447725
- SRA Study: SRP694115
- Assay Type: Amplicon Sequencing
- Host Organism: Homo sapiens
- Sample Origin: South Korea (Daegu)
- Sequencing Platform: Illumina MiSeq
- Library Layout: Paired-end
- Organism: Human gut metagenome
- Total Samples: 18
- Submission: School of Medicine, Keimyung University, JeongWoo Hwang; 2026-04-16

## Sample Types

Samples were collected from colorectal cancer patients and include:

- Tumor tissue samples
- Adjacent mucosal tissue samples
- Fecal samples

Patient identifiers include P01–P06, with multiple sample types available for most patients.

## Analysis Goals

Sequencing data from human colorectal cancer samples. The objective is to characterize microbial community composition and diversity across different sample types collected from colorectal cancer patients.

1. Import and validate raw sequencing data.
2. Perform quality assessment of paired-end reads.
3. Denoise sequences using DADA2.
4. Generate amplicon sequence variants (ASVs).
5. Evaluate alpha and beta diversity.
6. Assign taxonomy using a reference database.
7. Identify microbial taxa associated with colorectal cancer sample types. 


In [ ]:
## Creating Project Directories

Bioinformatics projects generate many intermediate and output files.

To keep the analysis organized, separate directories are created for:

- metadata: sample information
- sra: downloaded SRA files
- fastq: extracted sequencing reads
- qiime2: QIIME 2 artifacts and visualizations
- results: final outputs and figures

## Verify Current Working Directory

Before downloading files, verify that the notebook is operating within the project directory.

This ensures that all files are stored in the expected location.

In [4]:
!mkdir -p metadata
!mkdir -p sra
!mkdir -p fastq
!mkdir -p qiime2
!mkdir -p results
!pwd

/root/metagenomics


## Downloading Raw Sequencing Data AND Metadata

The raw paired-end FASTQ files and sample metadata corresponding to the colorectal cancer amplicon sequencing dataset will be downloaded from the NCBI Sequence Read Archive (SRA) using the SRA Toolkit.

The following SRA run accessions are included:
SRR38233223
SRR38233224
SRR38233225
SRR38233226
SRR38233227
SRR38233228
SRR38233229
SRR38233230
SRR38233231
SRR38233232
SRR38233233
SRR38233234
SRR38233235
SRR38233236
SRR38233237
SRR38233238
SRR38233239
SRR38233240


## Load Metadata Table

Metadata contains information about each sequencing sample.

Typical metadata fields include:

- Sample identifier
- SRA accession number
- Sample type
- Patient identifier
- Experimental group

Metadata is essential because QIIME 2 uses it for downstream statistical analysis and visualization.

In [8]:
import pandas as pd

metadata = pd.read_csv("metadata/SraRunTable.csv")
metadata.head()

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,LibraryLayout,LibrarySelection,LibrarySource,Organism,Platform,ReleaseDate,create_date,version,Sample Name,SRA Study
0,SRR38233223,AMPLICON,602,66362072,PRJNA1447725,SAMN57315478,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",38309070,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2021-12-17,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P04_feces,SRP694115
1,SRR38233224,AMPLICON,602,71240078,PRJNA1447725,SAMN57315477,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",40782819,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2021-10-11,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P03_tumor,SRP694115
2,SRR38233225,AMPLICON,602,71091986,PRJNA1447725,SAMN57315476,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",40980733,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2021-09-08,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P03_mucosa,SRP694115
3,SRR38233226,AMPLICON,602,66128496,PRJNA1447725,SAMN57315475,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",37987420,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2022-05-16,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P03_feces,SRP694115
4,SRR38233227,AMPLICON,602,57877484,PRJNA1447725,SAMN57315474,"MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut",36009445,"SCHOOL OF MEDICINE, KEIMYUNG UNIVERSITY",2021-09-14,...,PAIRED,PCR,METATRANSCRIPTOMIC,human gut metagenome,ILLUMINA,2026-04-23T00:00:00Z,2026-04-23T03:43:00Z,1,P02_tumor,SRP694115


## Downloading Raw Sequencing Data

The metadata table contains SRA accession numbers for all samples.

In this step we download the raw sequencing data from NCBI SRA.

The download occurs in two stages:

### 1. prefetch

Downloads the original SRA archive file from NCBI.

Example:

SRR38233223.sra

### 2. fasterq-dump

Converts the SRA archive into FASTQ format that can be processed by QIIME 2.

Because this dataset contains paired-end sequencing reads, each sample will generate:

- Forward reads (_1.fastq)
- Reverse reads (_2.fastq)

These FASTQ files will be used as input for the QIIME 2 workflow.

In [2]:
!prefetch --version
!fasterq-dump --version


prefetch : 3.2.1


fasterq-dump : 3.2.1



In [16]:
runs = metadata["Run"].tolist()

for run in runs:
    print(run)

SRR38233223
SRR38233224
SRR38233225
SRR38233226
SRR38233227
SRR38233228
SRR38233229
SRR38233230
SRR38233231
SRR38233232
SRR38233233
SRR38233234
SRR38233235
SRR38233236
SRR38233237
SRR38233238
SRR38233239
SRR38233240


In [17]:
import subprocess

runs = metadata["Run"].tolist()

for run in runs:
    print(f"Downloading {run}...")
    subprocess.run(
        ["prefetch", run, "--output-directory", "sra"],
        check=True
    )

print("Download complete.")

2026-06-07T05:57:37 prefetch.3.2.1: 1) Resolving 'SRR38233223'...
2026-06-07T05:57:42 prefetch.3.2.1: Current preference is set to retrieve SRA Normalized Format files with full base quality scores
2026-06-07T05:57:44 prefetch.3.2.1: 1) Downloading 'SRR38233223'...
2026-06-07T05:57:44 prefetch.3.2.1:  SRA Normalized Format file is being retrieved
2026-06-07T05:57:44 prefetch.3.2.1:  Downloading via HTTPS...
2026-06-07T05:59:10 prefetch.3.2.1:  HTTPS download succeed
2026-06-07T05:59:10 prefetch.3.2.1:  'SRR38233223' is valid: 38310989 bytes were streamed from 38297093
2026-06-07T05:59:10 prefetch.3.2.1: 1) 'SRR38233223' was downloaded successfully
2026-06-07T05:59:10 prefetch.3.2.1: 1) Resolving 'SRR38233223's dependencies...
2026-06-07T05:59:10 prefetch.3.2.1: 'SRR38233223' has 0 unresolved dependencies
2026-06-07T05:59:10 prefetch.3.2.1: 1) Resolving 'SRR38233224'...
2026-06-07T05:59:13 prefetch.3.2.1: Current preference is set to retrieve SRA Normalized Format files with full base q

## Converting SRA Archives to FASTQ Files

The downloaded SRA archives cannot be analyzed directly in QIIME 2.

Therefore, each SRA file is converted into FASTQ format using `fasterq-dump`.

Because this dataset was generated using paired-end sequencing, each sample will produce:

- Forward reads (`_1.fastq`)
- Reverse reads (`_2.fastq`)

These FASTQ files will be imported into QIIME 2 for downstream processing.

## Test Conversion of One Sample

Before converting all 18 samples, we first convert a single SRA archive.

This verifies that:

- The downloaded file is valid.
- Paired-end FASTQ files are generated correctly.
- Output files are written to the expected directory.

Testing a single sample helps identify problems before processing the full dataset.

In [21]:
!fasterq-dump \
sra/SRR38233223/SRR38233223.sra \
-O fastq

spots read      : 110,236
reads read      : 220,472
reads written   : 220,472


In [22]:
!ls -lh fastq | head

total 159M
-rw-r--r-- 1 root root 80M Jun  7 07:21 SRR38233223_1.fastq
-rw-r--r-- 1 root root 80M Jun  7 07:21 SRR38233223_2.fastq


In [23]:
import subprocess

remaining_runs = metadata["Run"].tolist()[1:]

for run in remaining_runs:
    print(f"Converting {run}...")
    
    subprocess.run([
        "fasterq-dump",
        f"sra/{run}/{run}.sra",
        "-O",
        "fastq"
    ], check=True)

print("All conversions completed.")

Converting SRR38233224...


spots read      : 118,339
reads read      : 236,678
reads written   : 236,678


Converting SRR38233225...


spots read      : 118,093
reads read      : 236,186
reads written   : 236,186


Converting SRR38233226...


...

spots read      : 115,076
reads read      : 230,152
reads written   : 230,152


Converting SRR38233239...


spots read      : 138,223
reads read      : 276,446
reads written   : 276,446


Converting SRR38233240...
All conversions completed.


spots read      : 125,021
reads read      : 250,042
reads written   : 250,042


## Compressing FASTQ Files

QIIME 2 manifest import expects gzipped FASTQ files.

The FASTQ files generated by `fasterq-dump` are currently uncompressed. Therefore, they are compressed using gzip before import.

In [34]:
!gzip fastq/*.fastq

## Data Download and Conversion Summary

All sequencing data were successfully downloaded from the NCBI Sequence Read Archive (SRA).

Results:

- Samples downloaded: 18
- SRA files: 18
- FASTQ files generated: 36 (18 + 18)
- Library layout: Paired-end
- Total FASTQ size: ~3 GB

The dataset is now ready for quality assessment and import into QIIME 2.

Downstream Analysis

## Importing Paired-End FASTQ Files into QIIME 2

The raw sequencing reads are currently stored as FASTQ files.

QIIME 2 requires these files to be imported into its native artifact format (`.qza`).

A manifest file is used to associate each sample ID with its forward and reverse FASTQ files.

The resulting artifact will contain all demultiplexed paired-end sequences and will serve as the starting point for downstream quality assessment and DADA2 denoising.

NOTE: Why PairedEndFastqManifestPhred33V2?
It tells QIIME 2 that you are importing paired-end Illumina FASTQ files using a manifest file, with quality scores encoded as Phred+33. The quality scores in the FASTQ files use Phred+33 encoding (standard for modern Illumina data).

In [35]:
import pandas as pd
from pathlib import Path

manifest = pd.DataFrame({
    "sample-id": metadata["Run"],
    "forward-absolute-filepath": [
        str(Path.cwd() / "fastq" / f"{run}_1.fastq.gz")
        for run in metadata["Run"]
    ],
    "reverse-absolute-filepath": [
        str(Path.cwd() / "fastq" / f"{run}_2.fastq.gz")
        for run in metadata["Run"]
    ]
})

manifest.to_csv(
    "metadata/manifest.tsv",
    sep="\t",
    index=False
)

In [37]:
!qiime tools import \
  --type 'SampleData[PairedEndSequencesWithQuality]' \
  --input-path metadata/manifest.tsv \
  --output-path qiime2/paired-end-demux.qza \
  --input-format PairedEndFastqManifestPhred33V2

Imported metadata/manifest.tsv as PairedEndFastqManifestPhred33V2 to qiime2/paired-end-demux.qza


In [38]:
!qiime tools peek qiime2/paired-end-demux.qza

UUID:        d966194a-6c07-497b-9c09-6c58f8ce5e43
Type:        SampleData[PairedEndSequencesWithQuality]
Data format: SingleLanePerSamplePairedEndFastqDirFmt


## Generating a Demultiplexed Sequence Summary

This step generates a summary visualization of the imported paired-end sequencing data.

### Why is this step performed?

The summary provides an overview of the sequencing dataset, including:

- Number of sequences obtained for each sample.
- Distribution of sequencing depth across samples.
- Forward read quality scores.
- Reverse read quality scores.
- Read length distribution.

### Why is this important?

The quality score plots are used to determine appropriate filtering and truncation parameters for DADA2 denoising.

By examining where sequence quality begins to decline, we can choose suitable trimming positions that maximize data retention while minimizing sequencing errors.

# Visualizing the demultiplexed sequences quality control graphs
### Output

The resulting visualization file (`.qzv`) can be viewed in QIIME 2 View and will guide the selection of DADA2 parameters in the next step of the analysis.

In [39]:
!qiime demux summarize \
  --i-data qiime2/paired-end-demux.qza \
  --o-visualization qiime2/paired-end-demux.qzv

Saved Visualization to: qiime2/paired-end-demux.qzv


## Denoising Sequences with DADA2

DADA2 is used to denoise the paired-end sequencing reads and infer Amplicon Sequence Variants (ASVs).

Based on the quality profiles generated in the previous step, reads are truncated to remove low-quality regions while retaining sufficient sequence length for successful read merging.

Truncation parameters selected:

- Forward reads: 240 bp
- Reverse reads: 220 bp

DADA2 performs:
- Quality filtering
- Error-rate learning
- Sequence denoising
- Paired-end read merging
- Chimera removal

The output will include a feature table containing ASV counts per sample and representative ASV sequences.

In [42]:
!qiime dada2 denoise-paired \
  --i-demultiplexed-seqs qiime2/paired-end-demux.qza \
  --p-trunc-len-f 240 \
  --p-trunc-len-r 220 \
  --o-table qiime2/table-dada2.qza \
  --o-representative-sequences qiime2/rep-seqs-dada2.qza \
  --o-denoising-stats qiime2/dada2-stats.qza \
  --o-base-transition-stats qiime2/base-transition-stats.qza

Saved FeatureTable[Frequency] to: qiime2/table-dada2.qza
Saved FeatureData[Sequence] to: qiime2/rep-seqs-dada2.qza
Saved SampleData[DADA2Stats] to: qiime2/dada2-stats.qza
Saved DADA2BaseTransitionStats to: qiime2/base-transition-stats.qza


## Creating a QIIME 2 Sample Metadata File

The original SRA metadata contains biological and technical information for each sample.

To enable sample annotation and downstream comparative analyses, the metadata are converted into a QIIME 2-compatible tab-separated metadata file.

The sample identifiers used in the metadata file must match the sample identifiers present in the QIIME 2 artifacts.

In [43]:
metadata.columns.tolist()

['Run',
 'Assay Type',
 'AvgSpotLen',
 'Bases',
 'BioProject',
 'BioSample',
 'BioSampleModel',
 'Bytes',
 'Center Name',
 'Collection_Date',
 'Consent',
 'DATASTORE filetype',
 'DATASTORE provider',
 'DATASTORE region',
 'env_broad_scale',
 'env_local_scale',
 'env_medium',
 'Experiment',
 'geo_loc_name_country',
 'geo_loc_name_country_continent',
 'geo_loc_name',
 'HOST',
 'Instrument',
 'lat_lon',
 'Library Name',
 'LibraryLayout',
 'LibrarySelection',
 'LibrarySource',
 'Organism',
 'Platform',
 'ReleaseDate',
 'create_date',
 'version',
 'Sample Name',
 'SRA Study']

In [14]:
import pandas as pd
import os

# Read original SRA metadata
df = pd.read_csv("/root/metagenomics/metadata/SraRunTable.csv")

# Create Patient_ID
df["Patient_ID"] = df["Sample Name"].str.extract(r"(P\d+)")

# Create Sample_Type
df["Sample_Type"] = df["Sample Name"].str.split("_").str[-1].str.capitalize()

# Select useful metadata columns
metadata = df[
    [
        "Run",
        "Sample Name",
        "Patient_ID",
        "Sample_Type",
        "HOST",
        "Organism",
        "geo_loc_name",
        "Collection_Date",
        "BioProject"
    ]
].copy()

# Rename Run column for QIIME2
metadata.rename(columns={"Run": "#SampleID"}, inplace=True)

# Save metadata file
metadata.to_csv(
    "/root/metagenomics/metadata/New_metadata.tsv",
    sep="\t",
    index=False
)

print("Metadata file saved successfully!")
print("\nColumns:")
print(metadata.columns.tolist())
print("\nShape:", metadata.shape)
print("\nPreview:")
print(metadata.head())

Metadata file saved successfully!

Columns:
['#SampleID', 'Sample Name', 'Patient_ID', 'Sample_Type', 'HOST', 'Organism', 'geo_loc_name', 'Collection_Date', 'BioProject']

Shape: (18, 9)

Preview:
     #SampleID Sample Name Patient_ID Sample_Type          HOST  \
0  SRR38233223   P04_feces        P04       Feces  Homo sapiens   
1  SRR38233224   P03_tumor        P03       Tumor  Homo sapiens   
2  SRR38233225  P03_mucosa        P03      Mucosa  Homo sapiens   
3  SRR38233226   P03_feces        P03       Feces  Homo sapiens   
4  SRR38233227   P02_tumor        P02       Tumor  Homo sapiens   

               Organism        geo_loc_name Collection_Date    BioProject  
0  human gut metagenome  South Korea: Daegu      2021-12-17  PRJNA1447725  
1  human gut metagenome  South Korea: Daegu      2021-10-11  PRJNA1447725  
2  human gut metagenome  South Korea: Daegu      2021-09-08  PRJNA1447725  
3  human gut metagenome  South Korea: Daegu      2022-05-16  PRJNA1447725  
4  human gut metagen

## Summarizing the DADA2 Feature Table

The DADA2 feature table contains the abundance of each Amplicon Sequence Variant (ASV) across all samples.

This step generates summary statistics describing:

- Number of detected ASVs.
- Total sequencing depth after denoising.
- Distribution of sequencing depth across samples.
- Feature frequencies and sample frequencies.

These summaries are used to evaluate the success of denoising and determine whether sufficient sequencing depth was retained for downstream analyses.

In [15]:
!qiime feature-table summarize \
  --i-table qiime2/table-dada2.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --o-feature-frequencies qiime2/feature-frequencies.qza \
  --o-sample-frequencies qiime2/sample-frequencies.qza \
  --o-summary qiime2/table-dada2.qzv

Saved ImmutableMetadata to: qiime2/feature-frequencies.qza
Saved ImmutableMetadata to: qiime2/sample-frequencies.qza
Saved Visualization to: qiime2/table-dada2.qzv


## Create rep-seqs-dada2.qzv

To inspect:

Number of ASVs
Sequence lengths
Actual nucleotide sequences
Later, associated taxonomy

What will we use these sequences for?
In a few steps, we'll classify them against a reference database (likely SILVA)

In [50]:
!qiime feature-table tabulate-seqs \
  --i-data qiime2/rep-seqs-dada2.qza \
  --o-visualization qiime2/rep-seqs-dada2.qzv

Saved Visualization to: qiime2/rep-seqs-dada2.qzv


## Evaluating DADA2 Read Retention

This step examines the number of reads retained after each stage of the DADA2 workflow.

The denoising statistics help identify where reads were lost during:

- Quality filtering
- Denoising
- Paired-end read merging
- Chimera removal

These metrics are used to evaluate whether the selected trimming parameters were appropriate and whether sufficient sequencing depth was retained for downstream analyses.

In [51]:
!qiime metadata tabulate \
  --m-input-file qiime2/dada2-stats.qza \
  --o-visualization qiime2/dada2-stats.qzv

Saved Visualization to: qiime2/dada2-stats.qzv


## Constructing a Phylogenetic Tree of ASVs

Many microbiome diversity metrics consider not only the presence and abundance of microorganisms but also their evolutionary relationships.

In this step, representative ASV sequences generated by DADA2 are aligned using MAFFT, a multiple sequence alignment algorithm. The aligned sequences are then used to construct a phylogenetic tree using FastTree.

The workflow generates:

- Multiple sequence alignment of ASVs
- Masked alignment with highly variable positions removed
- Unrooted phylogenetic tree
- Rooted phylogenetic tree

The rooted tree generated in this step will be used in subsequent diversity analyses.

In [52]:
!qiime phylogeny align-to-tree-mafft-fasttree \
  --i-sequences qiime2/rep-seqs-dada2.qza \
  --o-alignment qiime2/aligned-rep-seqs.qza \
  --o-masked-alignment qiime2/masked-aligned-rep-seqs.qza \
  --o-tree qiime2/unrooted-tree.qza \
  --o-rooted-tree qiime2/rooted-tree.qza

Saved FeatureData[AlignedSequence] to: qiime2/aligned-rep-seqs.qza
Saved FeatureData[AlignedSequence] to: qiime2/masked-aligned-rep-seqs.qza
Saved Phylogeny[Unrooted] to: qiime2/unrooted-tree.qza
Saved Phylogeny[Rooted] to: qiime2/rooted-tree.qza


### Phylogenetic Tree Visualization (optional)

The rooted phylogenetic tree generated using MAFFT and FastTree was exported in Newick format and visualized using iTOL (Interactive Tree Of Life) for interactive exploration of ASV phylogenetic relationships.

The phylogenetic tree served as the basis for Faith's Phylogenetic Diversity and UniFrac distance calculations used in downstream diversity analyses.

In [42]:
!qiime tools export \
  --input-path qiime2/rooted-tree.qza \
  --output-path rooted_tree_export

Exported qiime2/rooted-tree.qza as NewickDirectoryFormat to directory rooted_tree_export


## Core Diversity Analysis

To enable meaningful comparisons between samples, the feature table is rarefied to a uniform sequencing depth.

Based on the DADA2 feature table summary, a sampling depth of 3812 reads was selected because it corresponds to the lowest sequencing depth observed among all samples. This allows all 18 samples to be retained for downstream analyses.

Using the rooted phylogenetic tree and rarefied feature table, QIIME 2 calculates a set of alpha-diversity and beta-diversity metrics.

### Alpha Diversity Metrics

These metrics describe diversity within individual samples:

- Observed Features (ASV richness)
- Shannon Diversity Index
- Faith's Phylogenetic Diversity
- Evenness

### Beta Diversity Metrics

These metrics compare microbial community composition between samples:

- Jaccard Distance
- Bray-Curtis Distance
- Unweighted UniFrac Distance
- Weighted UniFrac Distance

Principal Coordinate Analysis (PCoA) plots are also generated to visualize similarities and differences among samples.

The resulting diversity metrics will be used to investigate differences between tumor, mucosa, and fecal microbiomes.

Based on current feature-table statistics:
Sampling depth = 3812 for core-metrics-phylogenetic
Max depth ≈ 6050 for alpha-rarefaction

## Alpha Rarefaction Analysis

Rarefaction analysis is performed to evaluate whether the sequencing depth is sufficient to capture the majority of microbial diversity present in the samples.

### Purpose
- Assess whether sequencing effort was adequate.
- Determine if diversity estimates have reached a plateau.
- Evaluate whether additional sequencing would likely reveal substantial new diversity.

A plateau in the rarefaction curves indicates that most microbial diversity has been captured and that sequencing depth is sufficient for downstream ecological analyses.

The maximum rarefaction depth is selected based on the observed sequencing depth distribution across samples while retaining as many samples as possible.

In [18]:
!qiime diversity alpha-rarefaction \
  --i-table qiime2/table-dada2.qza \
  --i-phylogeny qiime2/rooted-tree.qza \
  --p-max-depth 6050 \
  --m-metadata-file metadata/New_metadata.tsv \
  --o-visualization results/alpha-rarefaction.qzv

Saved Visualization to: results/alpha-rarefaction.qzv


## Alpha rarefaction interpretation

Alpha rarefaction analysis was performed to evaluate whether sequencing depth was sufficient to capture microbial diversity.

Observed feature richness continued to increase with sequencing depth, particularly in fecal samples, indicating that some rare taxa remained unsampled.

However, Shannon diversity curves reached a clear plateau at relatively low sequencing depths (~1000 reads), demonstrating stable estimates of community diversity and evenness.

Faith's Phylogenetic Diversity showed only minor increases beyond the selected rarefaction depth, suggesting that most phylogenetic diversity had been captured.

Based on these results, a rarefaction depth of 3812 reads per sample was selected for downstream diversity analyses, retaining all 18 samples while providing stable diversity estimates.

## Beta Diversity Analysis

Beta diversity analysis was performed using Bray-Curtis, Jaccard, Weighted UniFrac, and Unweighted UniFrac distance metrics.

Principal Coordinate Analysis (PCoA) revealed clustering patterns associated with sample type.

Statistical significance was assessed using PERMANOVA (Permutational Multivariate Analysis of Variance).

In [17]:
!qiime diversity core-metrics-phylogenetic \
  --i-phylogeny qiime2/rooted-tree.qza \
  --i-table qiime2/table-dada2.qza \
  --p-sampling-depth 3812  \
  --m-metadata-file metadata/New_metadata.tsv \
  --output-dir core-metrics-results_2

Saved FeatureTable[Frequency] to: core-metrics-results_2/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results_2/faith_pd_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results_2/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results_2/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results_2/evenness_vector.qza
Saved DistanceMatrix to: core-metrics-results_2/unweighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results_2/weighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results_2/jaccard_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results_2/bray_curtis_distance_matrix.qza
Saved PCoAResults to: core-metrics-results_2/unweighted_unifrac_pcoa_results.qza
Saved PCoAResults to: core-metrics-results_2/weighted_unifrac_pcoa_results.qza
Saved PCoAResults to: core-metrics-results_2/jaccard_pcoa_results.qza
Saved PCoAResults to: core-metrics-res

In [20]:
!qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results_2/bray_curtis_distance_matrix.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --m-metadata-column Sample_Type \
  --p-method permanova \
  --o-visualization results/bray-curtis-permanova.qzv

Saved Visualization to: results/bray-curtis-permanova.qzv


In [21]:
!qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results_2/jaccard_distance_matrix.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --m-metadata-column Sample_Type \
  --p-method permanova \
  --o-visualization results/jaccard_distance_matrix.qzv

Saved Visualization to: results/jaccard_distance_matrix.qzv


In [22]:
!qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results_2/unweighted_unifrac_distance_matrix.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --m-metadata-column Sample_Type \
  --p-method permanova \
  --o-visualization results/unweighted_unifrac_distance_matrix.qzv

Saved Visualization to: results/unweighted_unifrac_distance_matrix.qzv


In [23]:
!qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results_2/weighted_unifrac_distance_matrix.qza \
  --m-metadata-file metadata/New_metadata.tsv \
  --m-metadata-column Sample_Type \
  --p-method permanova \
  --o-visualization results/weighted_unifrac_distance_matrix.qzv

Saved Visualization to: results/weighted_unifrac_distance_matrix.qzv


## Beta Diversity Analysis Interpretation

Fecal samples consistently separated from mucosal and tumor-associated microbiomes across multiple distance metrics, indicating distinct microbial community structures.

The strongest separation was observed using Unweighted UniFrac and Jaccard distances, suggesting differences in community membership and phylogenetic composition between sample compartments.

Mucosal and tumor samples showed substantial overlap, indicating greater similarity in their microbial communities relative to fecal samples.

Statistical significance using PERMANOVA: Beta diversity analyses revealed significant differences in microbial community composition among fecal, mucosal, and tumor samples (Bray-Curtis p=0.013, Jaccard p=0.004, Unweighted UniFrac p=0.001), while Weighted UniFrac was not significant (p=0.170), suggesting that community differences are driven primarily by taxon membership and low-abundance phylogenetic lineages rather than dominant taxa.

## Taxonomic Classification 


Taxonomic Classification (Current Status)

Taxonomic classification has not yet been completed for this project.

The initial plan was to classify Amplicon Sequence Variants (ASVs) generated by DADA2 using a pre-trained Greengenes classifier (gg-13-8-99-515-806-nb-classifier.qza). However, the classifier was trained using scikit-learn 1.4.2, while the current QIIME 2 2026.4 environment uses scikit-learn 1.7.1.

## Future Work

Future updates to this repository may include:

Training a SILVA 138.2 classifier compatible with QIIME 2 2026.4.
Taxonomic assignment of ASVs.
Taxonomic composition analysis.
Differential abundance analysis between sample groups